# Arabic News Summarizer — quota-managed training

For an existing `full-v2-fp32-final-01` step-10 run, use **Arabic_News_Summarizer_Storage_Recovery.ipynb in a CPU runtime first**. Do not start GPU training or reuse the unsafe direct-to-Drive runner. This notebook now uses the same quota-managed profile for fresh runs.

The previous precision correction was user-verified on T4; this new storage profile is CPU-regression-tested, not yet verified live on Drive/T4. It uses local smoke saves, an immutable base, lossless full-state archives, actual account quota (including Trash), and confirmation-gated managed retention. No existing run artifacts or source locks are automatically deleted. See reports/STORAGE_RECOVERY.md for capacity and migration trade-offs.

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

import torch

print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU using Runtime > Change runtime type, then run all again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)


## Persistent storage and mode

The first run needs enough Drive space (40 GB project cap, with a 7 GB checkpoint reserve by default). Critical artifacts are written directly to Drive. Downloads and prepared data are cached there; temporary package installation and the code checkout live in `/content`.

Use the **same mode and RUN_DIR** on restart. Training settings are locked; only time/storage budgets may be raised. Use a new run directory for changed experiments. Never run two sessions against the same directory.

Set **MODE, CODE_REF, SMOKE_ONLY and RUN_NAME together in the next setup cell**. For the first retry before PR #2 is merged: `MODE="dev"`, `CODE_REF="563378c795da2ac483b9725c1c1d5486ef880622"`, `SMOKE_ONLY=True`, and a NEW run name such as `dev-v2-source-fixed-retry`. Later cells do not reset these settings. Do not use the old `full-v2-numerics` directory if it pinned pre-fix main.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# EDIT RETRY SETTINGS HERE ONLY. Later cells do not overwrite these variables.
MODE = os.environ.get("SUMMARY_MODE", "full")  # use "dev" for the first corrected smoke
CODE_REF = os.environ.get("SUMMARY_CODE_REF", "main")  # fixed SHA or PR branch before merge
SMOKE_ONLY = os.environ.get("SUMMARY_SMOKE_ONLY", "0") == "1"  # True for first retry
RUN_NAME = os.environ.get("SUMMARY_RUN_NAME", MODE + "-v2-numerics")
if MODE not in {"dev", "full"}:
    raise ValueError("SUMMARY_MODE must be dev or full")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a single safe directory name")
RUN_DIR = Path("/content/drive/MyDrive/ArabicNewsSummarizer") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "requested_source": CODE_REF, "smoke_only": SMOKE_ONLY,
       "persistent_run_directory": str(RUN_DIR)})
print("Preserve old experiments and source_commit.json files. Use a fresh directory for this correction.")


## Install the exact project version

On the first run, the source commit is frozen on Drive. Restarting restores that commit, even if the repository changes. The permanent default source ref is `main`. Before this correction is merged, set `CODE_REF` to the exact correction commit or PR branch. A new run fetches that ref even when a checkout already exists; a restart honors the saved commit. Dependencies are pinned where APIs are version-sensitive. No remote dataset Python script is executed.

In [ ]:
import ast
import json
import re
import sys


def checkout_source(repository, ref, repo_dir, run_dir, expected_schema=2):
    """Freeze an explicitly fetched ref; never reuse stale HEAD for a new run."""
    repo_dir, run_dir = Path(repo_dir), Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    new_checkout = not (repo_dir / ".git").exists()
    if new_checkout:
        subprocess.run(["git", "clone", "--no-checkout", repository, str(repo_dir)], check=True)
    origin = subprocess.check_output(["git", "-C", str(repo_dir), "remote", "get-url", "origin"], text=True).strip()
    if origin != repository:
        raise RuntimeError("Existing code checkout belongs to another repository; use a fresh runtime.")
    dirty = subprocess.check_output(["git", "-C", str(repo_dir), "status", "--porcelain"], text=True).strip()
    if dirty and not new_checkout:
        raise RuntimeError("Code checkout has local edits. Preserve them and use a fresh runtime.")
    source_lock = run_dir / "source_commit.json"
    locked = json.loads(source_lock.read_text()) if source_lock.exists() else None
    target = locked["commit"] if locked else ref
    if locked and (locked.get("repository") != repository or not re.fullmatch(r"[0-9a-f]{40}", target)):
        raise RuntimeError("Invalid source lock; preserve it and choose a new run directory.")
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", target], check=True)
    commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "FETCH_HEAD"], text=True).strip()
    if locked and commit != target:
        raise RuntimeError("Fetched commit does not match the persistent source lock.")
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", commit], check=True)
    # Validate the checked-out source without importing it or pinning an old main.
    config_path = repo_dir / "src/arabic_summarizer/config.py"
    schema = None
    if config_path.is_file():
        tree = ast.parse(config_path.read_text(encoding="utf-8"))
        for node in tree.body:
            if isinstance(node, ast.ClassDef) and node.name == "Config":
                for field in node.body:
                    if (isinstance(field, ast.AnnAssign) and isinstance(field.target, ast.Name)
                            and field.target.id == "training_schema_version"):
                        schema = ast.literal_eval(field.value)
    if schema != expected_schema or not (repo_dir / "src/arabic_summarizer/storage.py").exists():
        pin_status = "Existing source lock was preserved." if locked else "No new source lock was written."
        raise RuntimeError(
            f"Incompatible source: requested_ref={ref}, fetched_commit={commit}, "
            f"training_schema={schema}, expected={expected_schema}, config_file={config_path}. "
            f"{pin_status} The storage correction may not be merged into main. In the SETUP cell select the fixed "
            "branch codex/storage-safe-recovery and a NEW empty run directory. "
            "Do not delete or rewrite old source_commit.json files."
        )
    if not locked:
        temporary = source_lock.with_suffix(".tmp")
        temporary.write_text(json.dumps({"commit": commit, "repository": repository}))
        temporary.replace(source_lock)
    return commit


REPOSITORY = "https://github.com/nada-ali-13/Arabic-news-summarizer.git"
REPO_DIR = Path("/content/Arabic-news-summarizer")
if any(name == "arabic_summarizer" or name.startswith("arabic_summarizer.") for name in sys.modules):
    raise RuntimeError("Project modules are already loaded. Restart the Colab runtime to load the selected commit safely.")
commit = checkout_source(REPOSITORY, CODE_REF, REPO_DIR, RUN_DIR)
print("Source schema verified: 2; source commit:", commit)
print("Requested ref:", CODE_REF, "(an existing source_commit.json takes precedence)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-storage.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "installed_packages.txt").write_text(freeze)


## Execute or recover the complete plan

`dev`: two smoke steps and ten development optimizer steps. `full`: two smoke steps and two independent 1,200-step experiments (5e-5 and 1e-4), batch 1 with 16-step accumulation, maximum 1,024 source / 128 target tokens. Early stopping uses validation ROUGE-Lsum, patience 3. The smoke model is excluded from selection.

Evaluation uses the same fixed validation IDs across all candidates. The pretrained baseline is evaluated before real fine-tuning. The final test subset is untouched until the plan finishes. Arabic ROUGE uses Unicode word tokens rather than the default English-only tokenizer. Read the dataset statistics JSON and inspect saved qualitative examples.

The default session budget is 3 hours and the cumulative budget 7 hours; checks run at optimizer-step boundaries. Saving/evaluation may overrun a boundary. A paused plan is recoverable. If the cumulative budget is exhausted, raise `max_hours` below explicitly; experiments do not expand automatically. A numerical failure writes the exact failing phase, value, step and batch token evidence to `experiments/<name>/numerical_failure.json`. Do not infer precision as the cause from a generic traceback. Checks remain enabled in FP16 and FP32. T4 does not support BF16.

In [ ]:
import hashlib

from arabic_summarizer.config import load_config
from arabic_summarizer.drive_storage import DriveBackend, colab_service
from arabic_summarizer.io import atomic_json
from arabic_summarizer.recovery import StorageRecovery
from arabic_summarizer.training import run

config = load_config(REPO_DIR / "configs" / (MODE + ".json"))
# Optional budget change on restart, without changing experiment semantics:
# from dataclasses import replace
# config = replace(config, max_hours=10, session_hours=4)
assert getattr(config, "training_schema_version", None) == 2, "Old source pinned: use the fixed ref and a new run directory."
print(json.dumps(config.to_dict(), indent=2, ensure_ascii=False))

service = colab_service()
backend = DriveBackend(service, hashlib.sha256(str(RUN_DIR.resolve()).encode()).hexdigest()[:20])
controller = StorageRecovery(RUN_DIR, backend, Path("/content/arabic-storage") / backend.run_id, config=config)
config = controller.config
result = run(config, RUN_DIR, smoke_only=SMOKE_ONLY, storage=controller)
print(json.dumps(result, indent=2, ensure_ascii=False))
atomic_json(RUN_DIR / "notebook_last_result.json", result)


## Inspect actual results and qualitative samples

No scores are filled in ahead of execution. `final_report.json` exists only after completed training and final comparison. `best.json` can already point to a completed candidate while later training is paused. Review examples for invented facts, repeated sentences, omissions, fluency and incomplete outputs. The base denoising model may produce poor summaries before supervised transfer learning.

In [ ]:
from IPython.display import display

report_root = controller.control_root
for name in ["dataset_statistics.json", "experiments.json", "storage-plan.json", "final_report.json"]:
    path = report_root / name
    if path.exists():
        value = json.loads(path.read_text())
        print("\n", name)
        if name == "experiments.json":
            value = {k: {field: row.get(field) for field in ["status", "metrics", "global_step", "best_step"]}
                     for k, row in value.items()}
        display(value)
    else:
        print(name, "— pending")
if controller.store:
    display({"selected_drive_artifact": controller.store.index.get("best")})
examples_path = report_root / "best_test.json"
if examples_path.exists():
    examples = json.loads(examples_path.read_text())["examples"][:5]
    for example in examples:
        display({k: example[k] for k in ["id", "reference", "prediction"]})


## Launch the Arabic application (optional)

The best model/tokenizer are independent immutable export directories under `RUN_DIR/exports/`. `best.json` identifies the winner. Copy the selected export to `artifacts/best` for local use, or set `ARABIC_SUMMARIZER_MODEL` to its Drive path. No Hugging Face account is needed. Temporary Gradio sharing exposes the app via a public expiring link; run this cell only when you want to present it. Requests are handled one at a time to protect GPU memory.

In [ ]:
import gc

from app import build_app

best_artifact = controller.store.index.get("best") if controller.store else None
if best_artifact is None:
    print("No completed fine-tuned export yet. A smoke-only run does not create one; no app is launched.")
else:
    os.environ["ARABIC_SUMMARIZER_MODEL"] = controller.best_path()
    gc.collect()
    torch.cuda.empty_cache()
    demo = build_app()
    demo.launch(share=True, debug=False)


## Recovery checklist

1. Reopen this notebook, select a T4, and run all again. Authorize Drive.
2. Keep the same mode and run path. Frozen source/data/model revisions are restored.
3. Completed experiments are skipped; the newest valid full-state checkpoint resumes automatically.
4. A corrupt newest checkpoint falls back to an older valid one. If none survives, recovery fails clearly and preserves files for inspection. Use a new run directory only if restarting is intentional.
5. An immutable completed export stays available even when later experiments perform worse.

Drive itself is not a transactional filesystem across disconnections. Commit markers and hashes detect partial writes but cannot restore deleted/corrupted backups. Checkpoint SHA verification adds I/O time. Unsaved optimizer steps may be lost. Free Colab GPU availability, connection lifetime and final model quality are not guaranteed.

For the first retry of the numerical-safety correction, use the separate procedure in `reports/COLAB_RESTART.md`; do not edit or delete old `source_commit.json` files. Old training schemas are intentionally incompatible.
